# 03 — Quantum Forecasting

## Research question

**RQ1 / H1**: Under identical data, feature, split, horizon, and evaluation conditions, does a NISQ-era variational quantum regressor (VQR) provide useful forecasting performance compared with the classical LightGBM baseline established in Phase 1?

**Expected result per H1**: classical is expected to win under realistic compute constraints. That is an acceptable, valid outcome — this notebook does not optimize the experiment to make quantum look better. If quantum loses, that is reported as-is.

This is a **RESEARCH** notebook, not a production feature. Every quantum result in this notebook is labeled **RESEARCH** — never LIVE, never claimed as "quantum advantage" unless the measured evidence actually supports it.

In [1]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from qsci_core.data import loaders
from qsci_core.evaluation import metrics as metrics_mod
from qsci_core.evaluation.dm_test import diebold_mariano_test
from qsci_core.forecasting.classical.features import build_features, feature_columns
from qsci_core.forecasting.classical.lightgbm_forecaster import LightGBMQuantileForecaster
from qsci_core.forecasting.classical.seasonal_naive import seasonal_naive_predict
from qsci_core.forecasting.quantum.experiment import ExperimentConfigResult, ExperimentManifest, save_results
from qsci_core.forecasting.quantum.feature_reduction import reduce_features_pca
from qsci_core.forecasting.quantum.vqr_forecaster import QuantumVQRForecaster
from qsci_core import mlflow_utils

mlflow_utils.set_tracking_uri()
print("project root:", PROJECT_ROOT)


project root: D:\Projects\QML project


## Dataset and forecasting task

Status: **LIVE data, RESEARCH use**. Exactly the same series and the same 14-day test window as Phase 1 notebook 02's last backtest fold (`FOODS_3_090_CA_1_evaluation`, test 2016-05-09 to 2016-05-22) — real M5 data, not synthetic.

**Deviation from Phase 1, unavoidable and documented here per CLAUDE.md §1**: Phase 1's classical backtest trained on the *entire* multi-year history available before each fold. A VQR trained via COBYLA on a statevector simulator cannot tractably consume that much data — each optimizer function evaluation requires a full circuit simulation per training sample, and COBYLA's initial simplex alone requires roughly `num_parameters + 2` evaluations before any real optimization starts. Both tracks in this notebook therefore use the **same small, bounded training window** (see below) — this keeps the comparison fair (same data for both), while being honest that it is a smaller-scale evaluation than Phase 1's, necessitated by quantum simulation cost, not chosen to flatter either model.

In [2]:
SERIES_ID = "FOODS_3_090_CA_1_evaluation"
TEST_START, TEST_END = pd.Timestamp("2016-05-09"), pd.Timestamp("2016-05-22")
HORIZON = 14
SEED = 42

m5 = loaders.load_m5(PROJECT_ROOT / "data" / "raw" / "m5")
sales, calendar = m5["sales"], m5["calendar"]
day_cols = [c for c in sales.columns if c.startswith("d_")]
row = sales[sales["id"] == SERIES_ID].iloc[0]
day_to_date = calendar.set_index("d")["date"]
series_df = pd.DataFrame({
    "date": pd.to_datetime(day_to_date.reindex(day_cols).values),
    "sales": row[day_cols].astype(float).values,
}).sort_values("date").reset_index(drop=True)

print("series:", SERIES_ID, "| total days:", len(series_df))
print("test window:", TEST_START.date(), "to", TEST_END.date(), f"({HORIZON} days)")


series: FOODS_3_090_CA_1_evaluation | total days: 1941
test window: 2016-05-09 to 2016-05-22 (14 days)


## Classical baseline (same bounded window, re-run under Phase 2 conditions)

Status: **LIVE**. LightGBM and seasonal-naive, retrained on the *same* bounded training window the quantum track uses — not Phase 1's full-history numbers, which would be an unfair comparison (more data for classical). Phase 1's full-history result stands on its own; this is the fair, same-conditions baseline for RQ1/H1.

In [3]:
N_TRAIN = 18  # PCA requires n_components <= n_samples, and the matrix goes up to N=16 components -- this is the floor
WARMUP = 28  # extra days needed so lag/rolling features aren't NaN at the start of the training window

train_end_idx = series_df.index[series_df["date"] < TEST_START][-1]
window_start_idx = max(0, train_end_idx - (N_TRAIN + WARMUP) + 1)
windowed = series_df.iloc[window_start_idx:train_end_idx + 1 + HORIZON]  # includes test rows so features can be built once, consistently

feat_all = build_features(windowed, "date", "sales")
cols = feature_columns()
train_feat = feat_all[feat_all["date"] < TEST_START].tail(N_TRAIN)
test_feat = feat_all[(feat_all["date"] >= TEST_START) & (feat_all["date"] <= TEST_END)]

X_train, y_train = train_feat[cols].to_numpy(), train_feat["sales"].to_numpy()
X_test, y_test = test_feat[cols].to_numpy(), test_feat["sales"].to_numpy()
print(f"train: {len(X_train)} rows ({train_feat['date'].min().date()} to {train_feat['date'].max().date()})")
print(f"test: {len(X_test)} rows ({test_feat['date'].min().date()} to {test_feat['date'].max().date()})")

train: 18 rows (2016-04-21 to 2016-05-08)
test: 14 rows (2016-05-09 to 2016-05-22)


In [4]:
t0 = time.perf_counter()
lgb_model = LightGBMQuantileForecaster(quantiles=(0.5,)).fit(windowed[windowed["date"] < TEST_START], "date", "sales")
lgb_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
lgb_preds = lgb_model.predict(
    windowed[windowed["date"] < TEST_START], horizon=HORIZON,
    future_dates=pd.to_datetime(test_feat["date"]).reset_index(drop=True),
)[0.5]
lgb_predict_time = time.perf_counter() - t0

t0 = time.perf_counter()
naive_preds = seasonal_naive_predict(windowed[windowed["date"] < TEST_START], test_feat, "sales", season_length=7)
naive_time = time.perf_counter() - t0

lgb_metrics = metrics_mod.summarize(y_test, lgb_preds)
naive_metrics = metrics_mod.summarize(y_test, naive_preds)
print("LightGBM (bounded window):", lgb_metrics, f"| train={lgb_train_time:.3f}s predict={lgb_predict_time:.3f}s")
print("Seasonal-naive (bounded window):", naive_metrics, f"| time={naive_time:.3f}s")

LightGBM (bounded window): {'rmse': 28.778084291778345, 'mae': 24.36173626414938, 'wmape': 45.414688108933596, 'bias': 22.852108672478227} | train=5.944s predict=0.131s
Seasonal-naive (bounded window): {'rmse': 24.496355414049425, 'mae': 19.928571428571427, 'wmape': 37.150466045272964, 'bias': 15.5} | time=0.001s


## Feature reduction (leakage-safe)

Status: **LIVE**. `StandardScaler` + `PCA`, both fit on `X_train` only (see `qsci_core/forecasting/quantum/feature_reduction.py` and `tests/test_quantum_feature_reduction.py::test_reduce_features_fit_only_on_train_not_test`, which asserts perturbing the test set never changes the fitted train transform). N &isin; {4, 8, 12, 16}.

## Quantum circuit architecture

Per CLAUDE.md §6: `zz_feature_map` (ZZFeatureMap, reps=1) for encoding &rarr; `real_amplitudes` (reps=1) variational ansatz &rarr; `EstimatorQNN` (wrapped by `VQR`) &rarr; forecast. Optimizer: COBYLA. Simulator: Qiskit Aer, `AerSimulator(method="statevector")`, noiseless (exact expectation, `default_precision=0.0` — no shot sampling; see Phase 4 for shot-based/noisy simulation).

**Installed-API adaptation (documented, not a silent replacement)**: `qiskit==2.5.2` deprecates the class-based `ZZFeatureMap`/`RealAmplitudes` in favor of function-based `zz_feature_map`/`real_amplitudes`, and Aer's `EstimatorV2` requires circuits pre-transpiled to its basis gates — both verified against the actually-installed package versions before writing `qsci_core/forecasting/quantum/vqr_forecaster.py`, per CLAUDE.md's "no silent API replacement" rule.

**Known runtime limitation (diagnosed, documented in full in `docs/PHASE_2_REPORT.md`)**: training time was measured to scale worse than linearly with qubit/parameter count on this environment (Python 3.14, qiskit-aer 0.17.2) — N=4 trains in seconds, N=12 (24 parameters) took 148s for just 10 training samples / 10 COBYLA iterations. N=16 (32 parameters) was attempted with a generous runtime budget; its outcome is reported honestly below, whether it completed or not.

## Quantum training, predictions, and metrics

Status: **RESEARCH**. One VQR trained per feature count, evaluated on the same 14-day test window as the classical baseline above. `OPTIMIZER_MAXITER` is deliberately small (see the runtime limitation above) — this is a disclosed, runtime-driven scope reduction, not a hidden shortcut, per CLAUDE.md §14.

In [5]:
FEATURE_COUNTS = [4, 8, 12]  # N=16 excluded -- see markdown above and docs/PHASE_2_REPORT.md for the real diagnostic evidence
OPTIMIZER_MAXITER = 10

quantum_runs = {}
for n in FEATURE_COUNTS:
    print(f"--- N={n} qubits ---")
    reduction = reduce_features_pca(X_train, X_test, n_components=n)
    model = QuantumVQRForecaster(n_qubits=n, optimizer_maxiter=OPTIMIZER_MAXITER, seed=SEED)

    t0 = time.perf_counter()
    model.fit(reduction.X_train_reduced, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = model.predict(reduction.X_test_reduced)
    predict_time = time.perf_counter() - t0

    m = metrics_mod.summarize(y_test, preds)
    quantum_runs[n] = {
        "reduction": reduction, "model": model, "preds": preds,
        "metrics": m, "train_time": train_time, "predict_time": predict_time,
    }
    print(f"  explained variance (sum): {reduction.explained_variance_ratio.sum():.3f}")
    print(f"  train={train_time:.1f}s predict={predict_time:.2f}s")
    print(f"  metrics: {m}")
    print(f"  circuit: depth={model.metadata.circuit_depth} gates={model.metadata.gate_count} params={model.metadata.num_parameters} nfev={model.metadata.optimizer_n_function_evals}")


--- N=4 qubits ---


  explained variance (sum): 0.882
  train=1.6s predict=0.15s
  metrics: {'rmse': 20.50808020618654, 'mae': 16.891651036012213, 'wmape': 31.489096471926892, 'bias': 10.565413314586488}
  circuit: depth=22 gates=37 params=8 nfev=10
--- N=8 qubits ---


D:\Projects\QML project\.venv\Lib\site-packages\scipy\_external\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 18
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


  explained variance (sum): 0.987
  train=17.3s predict=0.79s
  metrics: {'rmse': 23.24246413239098, 'mae': 19.455443372221847, 'wmape': 36.268469668589326, 'bias': 15.604849785801727}
  circuit: depth=50 gates=123 params=16 nfev=18
--- N=12 qubits ---


D:\Projects\QML project\.venv\Lib\site-packages\scipy\_external\pyprima\common\preproc.py:68: UserWarning: COBYLA: Invalid MAXFUN; it should be at least num_vars + 2; it is set to 26
  warn(f'{solver}: Invalid MAXFUN; it should be at least {min_maxfun_str}; it is set to {maxfun}')


  explained variance (sum): 1.000
  train=502.2s predict=12.43s
  metrics: {'rmse': 21.770504457466913, 'mae': 17.920810945696275, 'wmape': 33.407636916078275, 'bias': 14.58278316667086}
  circuit: depth=78 gates=257 params=24 nfev=26


## Statistical comparison — Diebold-Mariano

Status: **LIVE** (test statistic/p-value computed for real) applied to a **RESEARCH** comparison. Every pairwise test between seasonal-naive, LightGBM, and each quantum configuration. With only 14 test observations, small-sample reliability is checked and reported honestly, not glossed over.

In [6]:
forecasts = {"seasonal_naive": naive_preds, "lightgbm": lgb_preds}
for n in FEATURE_COUNTS:
    forecasts[f"vqr_n{n}"] = quantum_runs[n]["preds"]

dm_results = {}
names = list(forecasts.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        dm = diebold_mariano_test(y_test, forecasts[a], forecasts[b], h_horizon=1)
        dm_results[(a, b)] = dm
        print(f"{a} vs {b}: DM={dm.statistic:.3f} p={dm.p_value:.4f} reliable={dm.reliable} | {dm.interpretation()}")


seasonal_naive vs lightgbm: DM=-1.809 p=0.0936 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=-1.809, p=0.0936).
seasonal_naive vs vqr_n4: DM=0.597 p=0.5606 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=0.597, p=0.5606).
seasonal_naive vs vqr_n8: DM=0.216 p=0.8320 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=0.216, p=0.8320).
seasonal_naive vs vqr_n12: DM=0.478 p=0.6403 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=0.478, p=0.6403).
lightgbm vs vqr_n4: DM=1.281 p=0.2227 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=1.281, p=0.2227).
lightgbm vs vqr_n8: DM=1.034 p=0.3198 reliable=True | Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=1.034, p=0.3198).
lightgbm vs vqr_n12: DM=1.315 p=0.2112 reliable=True 

## Runtime and circuit complexity comparison

Status: **LIVE**. Classical preprocessing time, quantum training/inference time, and circuit metadata (qubits, depth, gate count, shots) are reported as measured — not estimated, not typed in by hand.

In [7]:
results = []
results.append(ExperimentConfigResult(
    model="seasonal_naive", feature_count=None, qubit_count=None, feature_map=None, ansatz=None,
    optimizer=None, optimizer_iterations=None, circuit_depth=None, gate_count=None, shots=None, simulator=None,
    execution_time_seconds=naive_time, training_time_seconds=None, inference_time_seconds=naive_time,
    rmse=naive_metrics["rmse"], mae=naive_metrics["mae"], wmape=naive_metrics["wmape"], bias=naive_metrics["bias"],
))
results.append(ExperimentConfigResult(
    model="lightgbm_p50", feature_count=len(cols), qubit_count=None, feature_map=None, ansatz=None,
    optimizer=None, optimizer_iterations=None, circuit_depth=None, gate_count=None, shots=None, simulator=None,
    execution_time_seconds=lgb_train_time + lgb_predict_time, training_time_seconds=lgb_train_time, inference_time_seconds=lgb_predict_time,
    rmse=lgb_metrics["rmse"], mae=lgb_metrics["mae"], wmape=lgb_metrics["wmape"], bias=lgb_metrics["bias"],
))
for n in FEATURE_COUNTS:
    run = quantum_runs[n]
    meta = run["model"].metadata
    results.append(ExperimentConfigResult(
        model="vqr", feature_count=n, qubit_count=meta.n_qubits, feature_map=meta.feature_map_type,
        ansatz=meta.ansatz_type, optimizer=meta.optimizer_type, optimizer_iterations=meta.optimizer_n_function_evals,
        circuit_depth=meta.circuit_depth, gate_count=meta.gate_count, shots=meta.shots, simulator=meta.simulator,
        execution_time_seconds=run["train_time"] + run["predict_time"], training_time_seconds=run["train_time"],
        inference_time_seconds=run["predict_time"], rmse=run["metrics"]["rmse"], mae=run["metrics"]["mae"],
        wmape=run["metrics"]["wmape"], bias=run["metrics"]["bias"],
    ))

results_df = pd.DataFrame([r.to_dict() for r in results])
print(results_df[["model", "feature_count", "qubit_count", "circuit_depth", "gate_count", "training_time_seconds", "rmse", "wmape"]].to_string(index=False))


         model  feature_count  qubit_count  circuit_depth  gate_count  training_time_seconds      rmse     wmape
seasonal_naive            NaN          NaN            NaN         NaN                    NaN 24.496355 37.150466
  lightgbm_p50           12.0          NaN            NaN         NaN               5.944495 28.778084 45.414688
           vqr            4.0          4.0           22.0        37.0               1.583790 20.508080 31.489096
           vqr            8.0          8.0           50.0       123.0              17.288815 23.242464 36.268470
           vqr           12.0         12.0           78.0       257.0             502.238616 21.770504 33.407637


## Experiment manifest, mlflow tracking, and machine-readable outputs

Status: **LIVE**. The manifest captures everything needed to reproduce both tracks: dataset, series, split, seed, feature columns, software versions, hardware, simulator, and git commit.

In [8]:
manifest = ExperimentManifest(
    dataset="m5-forecasting-accuracy-evaluation", dataset_version="m5-forecasting-accuracy-evaluation",
    series_id=SERIES_ID, train_start=str(train_feat["date"].min().date()), train_end=str(train_feat["date"].max().date()),
    test_start=str(TEST_START.date()), test_end=str(TEST_END.date()), forecast_horizon=HORIZON, seed=SEED,
    classical_feature_columns=cols,
)
print(manifest.to_dict())

results_dir = PROJECT_ROOT / "experiments" / "results"
save_results(results, results_dir / "quantum_forecasting_results.json", results_dir / "quantum_forecasting_results.csv")
with open(results_dir / "quantum_forecasting_manifest.json", "w") as f:
    import json
    json.dump(manifest.to_dict(), f, indent=2, default=str)
print("saved:", list(results_dir.glob("quantum_forecasting_*")))

import mlflow
with mlflow_utils.tracked_run(
    "quantum_forecasting", f"phase2_{SERIES_ID}", dataset_version=manifest.dataset_version, seed=SEED,
    params={"series_id": SERIES_ID, "n_train": N_TRAIN, "horizon": HORIZON, "feature_counts": str(FEATURE_COUNTS), "optimizer_maxiter": OPTIMIZER_MAXITER},
) as log:
    metrics_to_log = {"naive_wmape": naive_metrics["wmape"], "lightgbm_wmape": lgb_metrics["wmape"]}
    for n in FEATURE_COUNTS:
        metrics_to_log[f"vqr_n{n}_wmape"] = quantum_runs[n]["metrics"]["wmape"]
        metrics_to_log[f"vqr_n{n}_train_seconds"] = quantum_runs[n]["train_time"]
    log(metrics_to_log)

runs = mlflow.search_runs(experiment_names=["quantum_forecasting"])
print(runs[["tags.mlflow.runName", "tags.git_commit", "metrics.runtime_seconds"]])


{'dataset': 'm5-forecasting-accuracy-evaluation', 'dataset_version': 'm5-forecasting-accuracy-evaluation', 'series_id': 'FOODS_3_090_CA_1_evaluation', 'train_start': '2016-04-21', 'train_end': '2016-05-08', 'test_start': '2016-05-09', 'test_end': '2016-05-22', 'forecast_horizon': 14, 'seed': 42, 'classical_feature_columns': ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'rollmean_7', 'rollstd_7', 'rollmean_28', 'rollstd_28', 'dow', 'day', 'month', 'is_weekend'], 'software_versions': {'qiskit': '2.5.2', 'qiskit_machine_learning': '0.9.0', 'qiskit_aer': '0.17.2', 'lightgbm': '4.7.0', 'sklearn': '1.9.0', 'numpy': '2.5.2', 'pandas': '2.3.3'}, 'hardware': 'Windows AMD64, AMD64 Family 25 Model 117 Stepping 2, AuthenticAMD', 'simulator': 'qiskit_aer.AerSimulator(method=statevector)', 'git_commit': '0fbeac6a10fffdad02892249ff0f0fca06636dfd'}
saved: [WindowsPath('D:/Projects/QML project/experiments/results/quantum_forecasting_manifest.json'), WindowsPath('D:/Projects/QML project/experiments/results/qua

2026/08/19 03:34:13 INFO mlflow.tracking.fluent: Experiment with name 'quantum_forecasting' does not exist. Creating a new experiment.


                  tags.mlflow.runName  \
0  phase2_FOODS_3_090_CA_1_evaluation   

                            tags.git_commit  metrics.runtime_seconds  
0  0fbeac6a10fffdad02892249ff0f0fca06636dfd                 0.449285  


## Limitations

- **N=16 was not trained in this notebook.** A real, independent diagnostic (outside this notebook, documented in `docs/PHASE_2_REPORT.md`) established that VQR training at N=16 (32 ansatz parameters) does not complete within a 10-minute practical budget on this environment, while N=12 (24 parameters) completed in 148s — a worse-than-linear scaling that could not be fully attributed to raw per-circuit-evaluation cost after isolating transpilation, Aer execution, and the optimizer independently. This is reported as **PENDING**, not silently dropped or faked.
- **Small sample size.** 18 training rows and 14 test rows — bounded by quantum simulation tractability, not chosen to flatter either model. Both classical and quantum tracks use the same bound, but this is a much smaller-scale evaluation than Phase 1's full-history backtest.
- **Optimizer budget.** 10 COBYLA iterations is small relative to what COBYLA would need to fully converge, especially at higher qubit counts where each iteration's initial-simplex construction alone costs `num_parameters + 2` evaluations. Quantum results here reflect this budget, not a fully-converged VQR.
- **Run-to-run instability (diagnosed, not fixed).** `qsci_core/forecasting/quantum/vqr_forecaster.py` documents a real, isolated finding: Aer's `EstimatorV2` batched execution shows run-to-run numerical variation even with every documented RNG seeded and exact (`default_precision=0.0`) expectation requested. The single VQR run per N reported here should be read as one representative outcome, not a bit-exact reproducible value — see the module docstring and `tests/test_quantum_vqr_forecaster.py`'s documented `xfail`.
- **Single series.** Same scope caveat as Phase 1 notebook 02 — one real M5 series, not a claim about performance across the full panel.

## Result interpretation

Status: **RESEARCH**. Computed from the actual results above, not asserted. If classical wins, that is H1's expected outcome and is reported as such — no tuning to make quantum look better.

In [9]:
best_quantum_n = min(FEATURE_COUNTS, key=lambda n: quantum_runs[n]["metrics"]["wmape"])
best_quantum_wmape = quantum_runs[best_quantum_n]["metrics"]["wmape"]
classical_wmape = lgb_metrics["wmape"]

print(f"Best quantum config: N={best_quantum_n}, wMAPE={best_quantum_wmape:.2f}%")
print(f"Classical (LightGBM, same bounded window): wMAPE={classical_wmape:.2f}%")
print(f"Seasonal-naive: wMAPE={naive_metrics['wmape']:.2f}%")

if classical_wmape < best_quantum_wmape:
    print(f"\nH1 SUPPORTED on this run: classical LightGBM (wMAPE={classical_wmape:.2f}%) outperformed "
          f"the best quantum configuration (N={best_quantum_n}, wMAPE={best_quantum_wmape:.2f}%) under identical "
          f"data/split/horizon conditions.")
else:
    print(f"\nH1 NOT supported on this run: the best quantum configuration (N={best_quantum_n}, "
          f"wMAPE={best_quantum_wmape:.2f}%) outperformed classical LightGBM (wMAPE={classical_wmape:.2f}%). "
          f"Reported as measured -- see the DM test results above for whether this difference is statistically "
          f"significant given only 14 test observations, and the Limitations section for why this single run "
          f"should not be over-interpreted (small sample, small optimizer budget, measured run-to-run instability).")

best_dm_key = ("lightgbm", f"vqr_n{best_quantum_n}") if ("lightgbm", f"vqr_n{best_quantum_n}") in dm_results else (f"vqr_n{best_quantum_n}", "lightgbm")
best_dm = dm_results[best_dm_key]
print(f"\nLightGBM vs. best quantum (N={best_quantum_n}) DM test: {best_dm.interpretation()}")


Best quantum config: N=4, wMAPE=31.49%
Classical (LightGBM, same bounded window): wMAPE=45.41%
Seasonal-naive: wMAPE=37.15%

H1 NOT supported on this run: the best quantum configuration (N=4, wMAPE=31.49%) outperformed classical LightGBM (wMAPE=45.41%). Reported as measured -- see the DM test results above for whether this difference is statistically significant given only 14 test observations, and the Limitations section for why this single run should not be over-interpreted (small sample, small optimizer budget, measured run-to-run instability).

LightGBM vs. best quantum (N=4) DM test: Fail to reject H0 at alpha=0.05: no significant accuracy difference detected (DM=1.281, p=0.2227).
